# **Step 1 : Load all research papers**

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

documents_path = "/content/drive/MyDrive/RAG_LLM_Project/Documents"

pdf_files = [
    os.path.join(documents_path, file)
    for file in os.listdir(documents_path)
    if file.lower().endswith(".pdf")
]

print("Number of papers:", len(pdf_files))
print("Papers:")

for file in pdf_files:
    print(os.path.basename(file))

Number of papers: 5
Papers:
RAG_survey.pdf
RAG_Comprehensive_survey.pdf
RAG_Evaluation_survey.pdf
RAGCHECKER.pdf
Improving_Text_Embeddings.pdf


# **Step 2 : Extract text fromt PDF**

This opens the PDF and allows Python to read its pages.
We first check the number of pages and then extract text.

In [3]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 1.7 MB/s eta 0:00:00


In [4]:
import os
from PyPDF2 import PdfReader

documents_path = "/content/drive/MyDrive/RAG_LLM_Project/Documents"

pdf_files = [
    os.path.join(documents_path, file)
    for file in os.listdir(documents_path)
    if file.lower().endswith(".pdf")
]

print("Number of papers:", len(pdf_files))

for file in pdf_files:
    print(os.path.basename(file))

Number of papers: 5
RAG_survey.pdf
RAG_Comprehensive_survey.pdf
RAG_Evaluation_survey.pdf
RAGCHECKER.pdf
Improving_Text_Embeddings.pdf


# **Step 3 : Extract text from all pages**

This extracts the text from every page of the research papers.
The extracted text is stored in documents so we can process it later.

In [5]:
documents = {}

for pdf_file in pdf_files:

    reader = PdfReader(pdf_file)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    documents[os.path.basename(pdf_file)] = text

print("Documents loaded:", len(documents))

for name, text in documents.items():
    print(name, "→", len(text), "characters")

Documents loaded: 5
RAG_survey.pdf → 109996 characters
RAG_Comprehensive_survey.pdf → 54001 characters
RAG_Evaluation_survey.pdf → 65394 characters
RAGCHECKER.pdf → 85968 characters
Improving_Text_Embeddings.pdf → 76449 characters


# **Step 4 : Clean Text and Create Chunks**

The paper is divided into smaller pieces called chunks so the RAG system can search them efficiently.
We use 500-word chunks with 100 words of overlap so important information isn't abruptly separated.

In [6]:
chunks = []
chunk_sources = []

chunk_size = 250
overlap = 50

for document_name, text in documents.items():

    words = text.split()

    # Remove the references section
    if "References" in words:
        references_index = words.index("References")
        words = words[:references_index]

    for i in range(0, len(words), chunk_size - overlap):

        chunk = words[i:i + chunk_size]

        if len(chunk) < 50:
            continue

        chunks.append(" ".join(chunk))
        chunk_sources.append(document_name)

print("Total chunks:", len(chunks))
print("Total sources:", len(chunk_sources))

Total chunks: 186
Total sources: 186


# **Step 5 : Create Text Embeddings**

Embeddings convert each text chunk into a numerical representation that captures its meaning.
These numbers allow the system to compare a user's question with the paper's chunks.

In [7]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

embeddings = embedding_model.encode(chunks)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (186, 384)


# **Step 6 : Creating FAISS Vector Index**

We are using FAISS, it is designed for fast similarity search of numerical values.
FAISS stores the chunk embeddings and allows us to search them quickly.
When a user asks a question, FAISS helps find chunks that are semantically similar.

In [8]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.8 MB/s eta 0:00:00


In [9]:
import faiss

In [10]:
index = faiss.IndexFlatL2(384)
index.add(embeddings)

In [11]:
index.ntotal

186

# **Step 7 : Load Cross-Encoder Reranker**

The reranker takes the initially retrieved chunks and examines how relevant each one is to the question.
This helps us select the most relevant chunk before sending information to the LLM.

In [12]:
!pip install sentence-transformers

In [13]:
from sentence_transformers import CrossEncoder

In [14]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

# **Step 8 : Load Qwen Language Model**

Qwen is the language model that generates the final answer.
It receives the retrieved information from the research paper and uses it to answer the user's question.

In [15]:
!pip install -q transformers accelerate

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "Qwen/Qwen2.5-0.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# **Step 9 : Build the RAG Pipeline**

This function connects all the major RAG components together: embedding, retrieval, reranking, and generation.
Instead of writing the entire process every time, we can simply call ask_rag(question).


In [25]:
def ask_rag(question):

    # Step 1: Convert question into an embedding
    query_embedding = embedding_model.encode([question])

    # Step 2: Retrieve relevant chunks
    distances, indices = index.search(query_embedding, 17)

    # Step 3: Rerank the retrieved chunks
    pairs = [(question, chunks[i]) for i in indices[0]]
    scores = reranker.predict(pairs)


    # Step 4: Find the best retrieval score
    best_score = max(scores)

    # Step 5: Check if the retrieved information is relevant
    if best_score < -5:
        return "The provided context does not contain enough information."

    # Step 6: Sort chunks by reranker score
    reranked = sorted(
        zip(scores, indices[0]),
        reverse=True
    )


    # Step 7: Select the best chunk from the correct source
    best_score, best_index = reranked[0]

    context = chunks[best_index]
    source = chunk_sources[best_index]

    print("Source:", source)
    print("Best retrieved context:")
    print(context[:3000])

    # Step 8: Give the context and question to Qwen
    messages = [
        {
            "role": "system",
            "content": """You are a research assistant answering questions about research papers.

Use only the information provided in the Context.

Do NOT copy sentences, bullet points, citations, or incomplete phrases directly from the Context.
Do NOT continue or reproduce the wording of the paper.
Instead, understand the information and write a new answer in your own words.

Answer the user's question directly and clearly in 3 to 4 complete sentences.
Include the most important information from the Context that helps answer the question.

If the Context does not contain enough information to answer the question, say:
"The provided context does not contain enough information." """
        },
        {
            "role": "user",
            "content": f"""Context:
{context}

Question:
{question}

Write the answer using only facts from the Context."""
        }
    ]

    # Step 9: Generate the answer
    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    # Step 10: Extract only the generated answer
    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = llm_tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return answer

# **Step 10 : Ask Questions to the RAG System**


The user can enter any question about the research paper.
The RAG system retrieves relevant information and generates an answer using that information.

In [26]:
question = input("Ask a question about the research papers: ")

answer = ask_rag(question)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(answer)

Ask a question about the research papers: what is RAGchecker?
Source: RAGCHECKER.pdf
Best retrieved context:
RAGCHECKER : A Fine-grained Framework for Diagnosing Retrieval-Augmented Generation Dongyu Ru1∗Lin Qiu1∗Xiangkun Hu1∗Tianhang Zhang1∗Peng Shi1∗ Shuaichen Chang1∗Cheng Jiayang1†Cunxiang Wang1†Shichao Sun2 Huanyu Li2Zizhao Zhang1†Binjie Wang1†Jiarong Jiang1Tong He1 Zhiguo Wang1Pengfei Liu2Yue Zhang3Zheng Zhang1 1Amazon AWS AI2Shanghai Jiaotong University3Westlake University Abstract Despite Retrieval-Augmented Generation (RAG) showing promising capability in leveraging external knowledge, a comprehensive evaluation of RAG systems is still challenging due to the modular nature of RAG, evaluation of long-form responses and reliability of measurements. In this paper, we propose a fine-grained evaluation framework, RAGCHECKER , that incorporates a suite of diagnostic metrics for both the retrieval and generation modules. Meta evaluation verifies that RAGCHECKER has significantly bette

In [27]:
%%writefile requirements.txt

sentence-transformers
faiss-cpu
transformers
accelerate
PyPDF2
torch

Writing requirements.txt


In [28]:
%%writefile README.md

# AI & RAG Research Assistant

An AI-powered research assistant that uses Retrieval-Augmented Generation (RAG) to answer questions from a collection of research papers.

## Features

- Loads multiple research papers in PDF format
- Extracts text from the papers
- Splits documents into overlapping chunks
- Converts chunks into vector embeddings using BGE
- Uses FAISS for similarity-based retrieval
- Uses a Cross-Encoder to rerank retrieved chunks
- Uses Qwen 2.5 0.5B Instruct to generate answers
- Refuses to answer when the retrieved context is not relevant

## RAG Pipeline

Documents → Chunking → Embeddings → FAISS Retrieval → Reranking → Qwen → Answer

## Technologies Used

- Python
- Sentence Transformers
- BGE embeddings
- FAISS
- Cross-Encoder
- Qwen 2.5
- PyPDF2
- Google Colab

## Dataset

The project uses five research papers related to Retrieval-Augmented Generation, text embeddings, and RAG evaluation.

## Example Questions

- What is RAGCHECKER?
- What are text embeddings used for?
- Why are text embeddings important for RAG?
- How does RAG reduce hallucinations in large language models?

## Out-of-Scope Handling

If the retrieved documents do not contain enough information to answer a question, the system responds:

"The provided context does not contain enough information."

## Limitations

- The current system uses a small language model (Qwen 2.5 0.5B).
- The system currently selects the highest-ranked retrieved chunk as context.
- PDF text extraction may not work perfectly for scanned documents.
- The project is currently designed for research-paper question answering.

## Future Improvements

- Add a web or Streamlit interface
- Support more document formats
- Improve document retrieval and context selection
- Add citation display for generated answers
- Add conversation history
- Deploy the application

## Author

Namra Aftab

Writing README.md


In [29]:
import os

for file in os.listdir():
    print(file)

.config
requirements.txt
drive
README.md
sample_data
